# Experiment 1: remove the CVSS score and prune the features that add nothing

**From the diagnosis** (`model_diagnosis.ipynb`): retraining the team model without `cvss_filled` raised 2025 hits from 19 to 24 (AP 0.125 → 0.135), and six features showed about zero importance on 2025. Those were single runs on 52 positives, below the ~6-hit detection limit, so they're leads, not results.

**Hypothesis:** the CVSS severity score encodes a pattern that doesn't carry over from one year to the next, and the near-zero features add estimation noise. A smaller model will rank new years better.

**Variants, fixed before running:**

| Variant | Logistic regression features | LightGBM features |
|---|---|---|
| **Current** (team model) | the team's 10 | the team's 14 |
| **P1: − CVSS** | 9 (no `cvss_filled`) | 13 |
| **P2: − CVSS − 6 near-zero** | `any_exploit`, `poc_github`, `nuclei`, `log_assigner_kev_rate`, `kw_unauth`, `ui_none` | the same + `log_batch_size` |

All model settings are the team's, unchanged.

**How it's judged:** the idea came from looking at 2025, so 2025 alone can't confirm it.
- **Primary:** the team's repeated grouped cross-validation (2024 + 2025, 85 positives, top 442), with a paired bootstrap against the current model, **also reported on the 2024 rows only**, which played no part in spotting the idea.
- **Secondary:** the forward check (2024 → 2025), and late 2025 (May-September) on its own.

**Decision rule, fixed in advance:**
- **Adopt (win):** the cross-validated gain has a 90% lower bound above 0, *and* the 2024 rows don't get worse.
- **Adopt (tie, simpler):** the cross-validated difference is ≥ 0, and the 2024 rows don't get worse. Then prefer the variant with fewer features, which is the team's own rule.
- **Otherwise:** keep the current model.

## Setup: the team model and data, rebuilt with the shared code

In [1]:
import sys, warnings; warnings.filterwarnings("ignore")
from pathlib import Path
ROOT = next(p for p in [Path(".."), Path(".")] if (p / "ml" / "config.py").exists()).resolve()
sys.path.insert(0, str(ROOT))
import numpy as np, pandas as pd, lightgbm as lgb
from scipy.stats import rankdata
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import average_precision_score
from sklearn.model_selection import StratifiedGroupKFold
from ml import config, data
from ml.featureset import FeatureBuilder
pd.set_option("display.width", 220)

con = data.connect()
train_rows, valid_rows = data.load_splits(con)
fb = FeatureBuilder(con)
tr, va = fb.transform(train_rows), fb.transform(valid_rows)

BASE = ["any_exploit", "poc_github", "nuclei", "log_assigner_kev_rate", "kw_unauth", "pr_none", "ui_none", "cvss_filled", "cwe_auth", "cwe_path"]
NEW4 = ["net_preauth", "log_batch_size", "log_cwe_kev_rate", "product_has_kev"]
NEAR_ZERO = ["pr_none", "net_preauth", "log_cwe_kev_rate", "product_has_kev", "cwe_auth", "cwe_path"]
drop = lambda cols, out: [c for c in cols if c not in out]
VARIANTS = {"Current": (BASE, BASE + NEW4),
            "P1: - CVSS": (drop(BASE, ["cvss_filled"]), drop(BASE + NEW4, ["cvss_filled"])),
            "P2: - CVSS - 6 near-zero": (drop(BASE, ["cvss_filled"] + NEAR_ZERO), drop(BASE + NEW4, ["cvss_filled"] + NEAR_ZERO))}
for k, (a, b) in VARIANTS.items():
    print(f"{k:26} LR {len(a):2} features | LightGBM {len(b):2} features")

GBM = dict(n_estimators=200, learning_rate=0.03, num_leaves=4, max_depth=2, min_child_samples=40, subsample=0.8, subsample_freq=1,
           colsample_bytree=0.8, reg_lambda=5, scale_pos_weight=50, verbose=-1)
def fit_lr(A, B, cols):
    return make_pipeline(StandardScaler(), LogisticRegression(C=0.1, class_weight="balanced", max_iter=5000)).fit(A[cols], A.y).predict_proba(B[cols])[:, 1]
def fit_gbm(A, B, cols, seed=0, n_seeds=3):
    return np.mean([lgb.LGBMClassifier(**GBM, random_state=seed * 10 + s).fit(A[cols], A.y).predict_proba(B[cols])[:, 1] for s in range(n_seeds)], axis=0)
blend = lambda a, b: 0.5 * rankdata(a) + 0.5 * rankdata(b)
def topk(y, s, frac=0.0069):
    k = int(round(len(y) * frac)); return int(np.asarray(y)[np.argsort(-np.asarray(s, dtype=float), kind="stable")[:k]].sum())
def paired_boot(y, a, b, frac=0.0069, n=2000, seed=0):
    y = np.asarray(y); k = int(round(len(y) * frac))
    ra, rb = rankdata(-np.asarray(a, float), method="ordinal"), rankdata(-np.asarray(b, float), method="ordinal")
    rng = np.random.default_rng(seed); d = np.empty(n)
    for i in range(n):
        idx = rng.integers(0, len(y), len(y))
        d[i] = y[idx][np.argpartition(ra[idx], k)[:k]].sum() - y[idx][np.argpartition(rb[idx], k)[:k]].sum()
    return topk(y, a, frac) - topk(y, b, frac), int(np.quantile(d, .05)), int(np.quantile(d, .95))

Current                    LR 10 features | LightGBM 14 features
P1: - CVSS                 LR  9 features | LightGBM 13 features
P2: - CVSS - 6 near-zero   LR  6 features | LightGBM  7 features


## 1. Primary test: repeated, grouped, stratified 5-fold cross-validation (the team's procedure)

In [2]:
L = pd.concat([tr.assign(year=2024), va.assign(year=2025)], ignore_index=True).sort_values("cve_id").reset_index(drop=True)
y = L.y.values; is24 = (L.year == 2024).values
groups = (L.assigner.fillna("?") + "|" + L.published.astype(str)).values
R = 5
o_lr = {v: np.zeros((R, len(L))) for v in VARIANTS}; o_gb = {v: np.zeros((R, len(L))) for v in VARIANTS}
for r in range(R):
    for f, (a, b) in enumerate(StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=r).split(L, y, groups)):
        A, B = L.iloc[a], L.iloc[b]
        for v, (c_lr, c_gb) in VARIANTS.items():
            o_lr[v][r, b] = fit_lr(A, B, c_lr); o_gb[v][r, b] = fit_gbm(A, B, c_gb, seed=r * 100 + f)
# blend after pooling every fold's probabilities (ranking all rows together)
oof = {v: np.array([blend(o_lr[v][r], o_gb[v][r]) for r in range(R)]) for v in VARIANTS}
avg = {v: o.mean(axis=0) for v, o in oof.items()}
rows = []
for v, o in oof.items():
    rows.append({"variant": v, "all rows: hits@442 (mean)": np.mean([topk(y, o[r]) for r in range(R)]),
                 "per repeat": [topk(y, o[r]) for r in range(R)],
                 "2024 rows: hits@199": np.mean([topk(y[is24], o[r][is24]) for r in range(R)]),
                 "2025 rows: hits@243": np.mean([topk(y[~is24], o[r][~is24]) for r in range(R)]),
                 "AP (all)": round(np.mean([average_precision_score(y, o[r]) for r in range(R)]), 4),
                 "AP (2024 rows)": round(np.mean([average_precision_score(y[is24], o[r][is24]) for r in range(R)]), 4)})
cv = pd.DataFrame(rows).set_index("variant"); cv

,all rows: hits@442 (mean),per repeat,2024 rows: hits@199,2025 rows: hits@243,AP (all),AP (2024 rows)
variant,,,,,,
Current,35.6,"[35, 35, 37, 35, 36]",16.8,19.4,0.0989,0.0766
P1: - CVSS,34.4,"[33, 29, 39, 36, 35]",15.4,20.4,0.0936,0.0653
P2: - CVSS - 6 near-zero,31.4,"[32, 30, 31, 32, 32]",16.0,18.2,0.0966,0.0636


**Reading this (cross-validation, the primary test):** the current model scores highest overall (35.6 of 85 in the top 442) and has the best average precision, both overall (0.099) and on the 2024 rows (0.077). Without CVSS (P1), it scores 34.4, but its repeats swing widely (29-39): the result became *less* stable. P2 scores 31.4.

In [3]:
tests = []
for v in ["P1: - CVSS", "P2: - CVSS - 6 near-zero"]:
    d, lo, hi = paired_boot(y, avg[v], avg["Current"])
    d24, lo24, hi24 = paired_boot(y[is24], avg[v][is24], avg["Current"][is24])
    tests.append({"variant": v, "all rows: diff (top 442)": f"{d:+d} ({lo:+d} to {hi:+d})",
                  "2024 rows: diff (top 199)": f"{d24:+d} ({lo24:+d} to {hi24:+d})", "lo": lo, "d": d, "d24": d24})
tests = pd.DataFrame(tests).set_index("variant")
tests[["all rows: diff (top 442)", "2024 rows: diff (top 199)"]]

,all rows: diff (top 442),2024 rows: diff (top 199)
variant,,
P1: - CVSS,+0 (-4 to +4),-1 (-4 to +2)
P2: - CVSS - 6 near-zero,-3 (-8 to +3),+1 (-3 to +5)


**Reading this:** in the paired test, P1 is +0 (range −4 to +4) and P2 is −3 (range −8 to +3). On the 2024 rows, which played no part in spotting the idea, P1 is −1 and P2 is +1. **Neither pruned version beats the current model in cross-validation.**

## 2. Secondary: forward check (train 2024 → test 2025) and late 2025 on its own

In [4]:
A24, B25 = L[is24], L[~is24]
late = (B25.snap >= "2025-05").values
fwd = {}
for v, (c_lr, c_gb) in VARIANTS.items():
    p_lr = fit_lr(A24, B25, c_lr)
    p_gb = np.mean([fit_gbm(A24.iloc[np.random.default_rng(s).permutation(len(A24))], B25, c_gb, seed=s, n_seeds=1) for s in range(20)], axis=0)
    fwd[v] = blend(p_lr, p_gb)
base25 = (B25.any_exploit * 10 + B25.epss.fillna(0)).values
rows = [{"variant": "Baseline: exploit code then EPSS", "2025 hits@243": topk(B25.y, base25),
         "May-Sep 2025 hits@133": topk(B25.y.values[late], base25[late])}]
for v, s in fwd.items():
    r = {"variant": v, "2025 hits@243": topk(B25.y, s), "2025 AP": round(average_precision_score(B25.y, s), 4),
         "May-Sep 2025 hits@133": topk(B25.y.values[late], s[late])}
    if v != "Current":
        d, lo, hi = paired_boot(B25.y.values, s, fwd["Current"]); r["vs. current (2025)"] = f"{d:+d} ({lo:+d} to {hi:+d})"
    rows.append(r)
pd.DataFrame(rows).set_index("variant").fillna("")

,2025 hits@243,May-Sep 2025 hits@133,2025 AP,vs. current (2025)
variant,,,,
Baseline: exploit code then EPSS,14,8,,
Current,18,8,0.1272,
P1: - CVSS,24,10,0.1344,+6 (+1 to +11)
P2: - CVSS - 6 near-zero,22,11,0.1317,+4 (-1 to +9)


**Reading this (forward check):** on 2025, the CVSS-free model does score higher: 24 vs. 18, a paired +6 (range +1 to +11). P2 scores 22. That **reproduces the diagnosis's single-run finding**, but 2025 is the year the idea came from, so this test can't confirm it independently. Late 2025 (May-September): current 8, P1 10, P2 11, baseline 8.

## 3. Decision, by the rule fixed above

In [5]:
decision = {}
for v in tests.index:
    t = tests.loc[v]
    if t.lo > 0 and t.d24 >= 0: decision[v] = "WIN"
    elif t.d >= 0 and t.d24 >= 0: decision[v] = "TIE (simpler)"
    else: decision[v] = "LOSE"
    print(f"{v}: CV {t['all rows: diff (top 442)']}, 2024 rows {t['2024 rows: diff (top 199)']} -> {decision[v]}")
wins = [v for v, d in decision.items() if d == "WIN"]
ties = [v for v, d in decision.items() if d == "TIE (simpler)"]
chosen = (max(wins, key=lambda v: tests.loc[v, "d"]) if wins else
          (min(ties, key=lambda v: len(VARIANTS[v][0])) if ties else "Current"))
print("\nChosen:", chosen)

P1: - CVSS: CV +0 (-4 to +4), 2024 rows -1 (-4 to +2) -> LOSE
P2: - CVSS - 6 near-zero: CV -3 (-8 to +3), 2024 rows +1 (-3 to +5) -> LOSE

Chosen: Current


**Decision: keep the current model.** By the rule fixed in advance, both pruned versions lose: neither shows a cross-validated gain, and P1 is slightly worse on the 2024 rows.

**What this means:** removing CVSS helps on 2025 but not when 2024 and 2025 are mixed in cross-validation. The likely reading is that CVSS's value **changes from year to year** (in 2025 the model does better without it), not that it's useless everywhere. Experiment 2's strictly time-ordered test (`exp2_rolling_forward_test.ipynb`) looks at this from another angle. Read the two together before deciding.

In [6]:
sb = fb.transform(__import__("features").scoreboard_frame(con).sort_values("cve_id").reset_index(drop=True))
def final_top500(c_lr, c_gb):
    p_lr = make_pipeline(StandardScaler(), LogisticRegression(C=0.1, class_weight="balanced", max_iter=5000)).fit(L[c_lr], L.y).predict_proba(sb[c_lr])[:, 1]
    p_gb = np.mean([lgb.LGBMClassifier(**GBM, random_state=s).fit(L.iloc[np.random.default_rng(s).permutation(len(L))][c_gb],
                                                                    L.iloc[np.random.default_rng(s).permutation(len(L))].y).predict_proba(sb[c_gb])[:, 1]
                    for s in range(30)], axis=0)
    return sb.assign(score=0.5 * rankdata(p_lr) / len(sb) + 0.5 * rankdata(p_gb) / len(sb)).nlargest(500, "score")
team_top = final_top500(*VARIANTS["Current"])
if chosen != "Current":
    new_top = final_top500(*VARIANTS[chosen])
    out = ROOT / "outputs"; out.mkdir(exist_ok=True)
    assert len(new_top) == 500 and new_top.cve_id.is_unique and new_top.cve_id.isin(sb.cve_id).all()
    new_top[["cve_id"]].to_csv(out / "top500_pruned.csv", index=False)
    print(f"{chosen}: top 500 shares {new_top.cve_id.isin(team_top.cve_id).sum()} CVEs with the current model's list; "
          f"exploit code {new_top.any_exploit.mean():.0%} vs. {team_top.any_exploit.mean():.0%}. Wrote outputs/top500_pruned.csv")
else:
    print("Current model kept; no new list written.")

Current model kept; no new list written.
